# OpenAI Agents SDK

Merged walkthrough of Week 2 labs:
- [1_lab1.ipynb](https://github.com/ed-donner/agents/blob/main/2_openai/1_lab1.ipynb) — Agent, Runner, traces, tools, sessions
- [2_lab2.ipynb](https://github.com/ed-donner/agents/blob/main/2_openai/2_lab2.ipynb) — multi-agent orchestration (code + LLM)

**Docs:** https://openai.github.io/openai-agents-python/

**Before running:** parent `.env` with `OPENAI_API_KEY`. Optional: `PUSHOVER_USER` / `PUSHOVER_TOKEN` for notify tools later.

**Package name note:** install `openai-agents` (not the unrelated `agents` package on PyPI).


In [2]:
# install openai-agents in command line
!pip3 install openai-agents

  Using cached pydantic-2.13.5-py3-none-any.whl.metadata (110 kB)
  Using cached pydantic_core-2.46.5-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.3 kB/s  0:00:30 eta 0:00:03
Using cached pydantic-2.13.5-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.5-cp313-cp313-macosx_11_0_arm64.whl (1.9 MB)
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.33.2
    Uninstalling pydantic_core-2.33.2:
      Successfully uninstalled pydantic_core-2.33.2
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.11.7
    Uninstalling pydantic-2.11.7:
      Successfully uninstalled pydantic-2.11.7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [openai-agents]0m [openai-agents]


In [3]:
# imports + env
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

## Practice arc

Work the stages in order. Each builds on the last.

| Stage | Focus | Checkpoint |
|------|--------|------------|
| 1 | SDK hello — `Agent` + `Runner` | One agent reply works |
| 2 | Observability + streaming | Trace on [OpenAI traces](https://platform.openai.com/traces) |
| 3 | Function tools (`@function_tool`) | Tool call visible in the trace |
| 4 | Memory / sessions | Name remembered across two `Runner.run` calls |
| 5 | Orchestrate by code | Parallel drafts → picker (optional send) |
| 6 | Orchestrate by LLM | Manager via agents-as-tools, then handoffs |

Delivery (email/SMTP) is optional. Prefer Pushover or a dry-run print tool so the focus stays on agents.


---
## Stage 1 — SDK hello

A simple `Agent` plus `Runner.run` is enough for a first loop: name, instructions, model, then a prompt.

Practice:
1. Create an agent (e.g. a joke teller)
2. `await Runner.run(agent, "...")`
3. Read `result.final_output` and `result.to_input_list()` to see the turn detail


In [ ]:
# Stage 1 — Agent + Runner.run

# setup agent
MODEL = "gpt-4o-mini"
instructions = "You are a helpful assistant that can answer questions and help with tasks."

agent = Agent(name="assistant", instructions=instructions, model=MODEL)

# run agent with question
user_input = "please help me solve this math problem: 2 + 2 = ?"
result = await Runner.run(agent, user_input)
print(result.final_output)
result.to_input_list()


The solution to the problem \(2 + 2\) is \(4\).


[{'content': 'please help me solve this math problem: 2 + 2 = ?',
  'role': 'user'},
 {'id': 'msg_056a47467b698aa2006aac182a9ee487d08ec18b9a13e97c04',
  'content': [{'annotations': [],
    'text': 'The solution to the problem \\(2 + 2\\) is \\(4\\).',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

---
## Stage 2 — Observability + streaming

### Trace (observability)

Wrap a run in `trace("...")` and open the OpenAI dashboard to inspect the call graph.

**Checkpoint:** find this run under https://platform.openai.com/traces

### Streaming — why it exists

`Runner.run` waits until the model finishes, then gives you the full answer in `result.final_output`. That is simple, but the notebook (or UI) looks idle until everything is done.

`Runner.run_streamed` starts the same agent loop, but **emits events as tokens arrive**. You can print (or render) text piece by piece — the same idea as ChatGPT typing live.

### How the stream cell works

1. **`Runner.run_streamed(agent, input=...)`** — kick off the run; returns a streamed result handle (do not `await` this call itself).
2. **`async for event in result.stream_events()`** — iterate every event from the run (raw model chunks, tool activity, etc.).
3. **Filter to text deltas** — keep events where `event.type == "raw_response_event"` and `event.data` is a `ResponseTextDeltaEvent`.
4. **`print(event.data.delta, end="", flush=True)`** — each `delta` is a small string fragment; concatenate them in order to rebuild the reply as it streams.

`ResponseTextDeltaEvent` comes from `openai.types.responses` — it is the SDK’s typed wrapper for “new text chunk” events on the Responses API stream.

### Contrast with Stage 1

| | `Runner.run` | `Runner.run_streamed` |
|--|--------------|------------------------|
| When you see output | After the full reply | Token by token |
| Typical access | `result.final_output` | `event.data.delta` in a loop |
| Best for | Scripts, tests, batch | Chat UIs, demos, long answers |

Same agent, same model — only the delivery of tokens changes.

### Cost note

Streaming does **not** mean more LLM calls or more agent-loop iterations. `Runner.run` and `Runner.run_streamed` do the same generation for that turn; streaming only changes how tokens are delivered over the wire.

You are billed for **input + output tokens** (and any tool rounds), not per stream event or per `delta`. Local `async for` / `print` loops are free — they just consume the stream.

Cost goes up when the answer is longer, you add tool/agent rounds, or you re-run the cell — same as without streaming.


In [ ]:
# Stage 2 — trace
user_input = "please help me solve this math problem: 2 + 2 = ?"
with trace("assitance"):

    result = await Runner.run(agent, user_input)
print(result.final_output)


The answer to \(2 + 2\) is \(4\).


In [10]:
# Stage 2 — stream
from openai.types.responses import ResponseTextDeltaEvent

user_input = "Please tell me 5 jokes about AI Agents."

result = Runner.run_streamed(agent, input=user_input)
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)


Sure! Here are five jokes about AI agents:

1. **Why did the AI agent break up with its data?**  
   It just couldn’t find the right connection!

2. **How does an AI agent stay warm?**  
   It turns on its “byte” heater!

3. **Why did the AI go to therapy?**  
   It had too many unresolved variables!

4. **What did the AI say when it won an award?**  
   “This is the best kind of recognition I’ve ever processed!”

5. **Why did the AI become a musician?**  
   Because it had great algorithms for composing! 

I hope you enjoyed them!

---
## Stage 3 — Function tools

Use `@function_tool` so the SDK builds the JSON schema for you. Attach the tool to an agent and ask it to notify (Pushover) or print a dry-run message.

Practice:
1. Define a small notify / push helper
2. Decorate it with `@function_tool`
3. Inspect `.params_json_schema`
4. Run an agent that must call the tool; confirm in the trace


In [ ]:
# Stage 3 — @function_tool + agent with tools
# @function_tool
# def notify_user(message: str) -> str:
#     ...
# notify_user.params_json_schema
# agent = Agent(..., tools=[notify_user])
# with trace("..."):
#     result = await Runner.run(agent, "...")


---
## Stage 4 — Memory / sessions

Each `Runner.run` starts fresh unless you pass history.

Practice both approaches:
1. **Manual** — feed the next turn with `result.to_input_list()` plus the new user message
2. **Built-in** — `SQLiteSession` (in-memory is fine for the lab)

**Checkpoint:** after “Hi, my name is …”, a second run answers with that name.


In [ ]:
# Stage 4a — no memory (expect failure to recall)
# ...


In [ ]:
# Stage 4b — manual history via to_input_list()
# ...


In [ ]:
# Stage 4c — SQLiteSession
# session = SQLiteSession("...")
# await Runner.run(agent, "...", session=session)
# await Runner.run(agent, "What's my name?", session=session)


---
## Stage 5 — Orchestrate by code

More predictable than letting one LLM plan everything: you wire the workflow in Python.

Pattern (sales cold-email theme from lab2, or any “N drafts → pick best”):
1. Define 3 writer agents with different styles
2. Run them in parallel with `asyncio.gather`
3. Pass all drafts to a picker agent
4. Optionally attach a send/notify tool and require a tool call (`ModelSettings(tool_choice="required")`)

**Checkpoint:** one trace shows parallel writers + selection (and optional send).


In [ ]:
# Stage 5a — three writer agents + streamed sample
# ...


In [ ]:
# Stage 5b — asyncio.gather parallel runs
# import asyncio
# with trace("Parallel drafts"):
#     results = await asyncio.gather(
#         Runner.run(writer1, message),
#         Runner.run(writer2, message),
#         Runner.run(writer3, message),
#     )


In [ ]:
# Stage 5c — picker (+ optional send tool)
# with trace("Selection workflow"):
#     ...
#     best = await Runner.run(picker, drafts)
#     # or sender agent with tools=[send_tool]


---
## Stage 6 — Orchestrate by LLM

Two collaboration styles in the OpenAI Agents SDK:

### 6a — Agents as tools
Expose writers with `agent.as_tool(...)`. A **Sales Manager** (planning agent) calls writer tools, picks the best draft, then calls send/notify.

Flow: **A → B → A** (control returns to the manager).

### 6b — Handoffs
Manager drafts via tools, then **hands off** to a sender. Control passes across: **A → B**.

Handoffs can be flaky—especially on smaller models. Iterate on prompts, or prefer agents-as-tools when you need reliability.

Optional: `draw_graph(manager)` to sketch the graph.

**Checkpoint:** compare the two traces; confirm only one outbound message when send is enabled.


In [ ]:
# Stage 6a — writers as tools + manager
# tool1 = writer1.as_tool(tool_name="...", tool_description="...")
# manager = Agent(name="Sales Manager", instructions="...", tools=[tool1, tool2, tool3, send_tool], ...)
# with trace("Sales manager (tools)"):
#     result = await Runner.run(manager, task)


In [ ]:
# Stage 6b — handoffs
# manager = Agent(..., tools=[tool1, tool2, tool3], handoffs=[sender], ...)
# with trace("Sales manager (handoffs)"):
#     result = await Runner.run(manager, task)


---
## Stretch (optional)

From the lab exercises — only after stages 1–6 feel solid:

- Rebuild a Week 1 idea (digital twin or checklist loop) on this SDK
- Swap Pushover/dry-run for SendGrid or Resend; handle a reply and keep an SDR conversation going
- Add another refine pass (draft → critique → rewrite) with either code orchestration or LLM orchestration

When you extract this notebook into a package, mirror `2_harness`: `agent/` modules + `tests/` with mocks, and a thin `openai_agent.py` entrypoint.
